# React 框架 · 第 04 天

## React Router 路由基础与记账本项目准备

本篇依据已下载的 React Router day04 笔记、PPT 和随堂代码整理。重点学习前端路由的基本概念、React Router 的配置和导航、路径参数与查询参数、嵌套路由、默认路由、404 兜底以及 history/hash 两种路由模式。

## 一、学习目标

学完本篇后，应能：

- 解释前端路由中 path 与 component 的对应关系。
- 安装并使用 `react-router-dom` 创建路由应用。
- 使用 `createBrowserRouter` 和 `RouterProvider` 配置路由表。
- 使用 `Link`、`NavLink` 和 `useNavigate` 实现声明式、编程式导航。
- 传递并读取路径参数、查询参数。
- 使用 `children` 和 `Outlet` 构建嵌套路由。
- 配置 index 默认子路由和 `*` 404 路由。
- 区分 history 模式和 hash 模式，为记账本项目搭好路由基础。

## 二、什么是前端路由

前端路由可以理解为：**一个 URL 路径对应一个页面组件**。当浏览器地址发生变化时，路由系统匹配对应的 path，并在当前页面渲染相应 component。

传统多页应用通常由服务器根据 URL 返回新的 HTML；单页应用则在首次加载后由前端路由切换页面内容，减少整页刷新。

```text
/login       → Login 登录组件
/article     → Article 文章组件
/about       → About 关于组件
```

路由系统通常还要解决：

- 页面之间如何跳转。
- 跳转时如何携带 id、筛选条件等参数。
- 公共布局如何复用。
- 找不到路径时如何展示 404 页面。
- 刷新页面时服务器如何配合。

## 三、创建 React Router 开发环境

课件使用 CRA 创建项目并安装 React Router：

```bash
npx create-react-app react-router-pro
cd react-router-pro
npm i react-router-dom
npm run start
```

推荐把路由配置单独放在 `src/router/index.js`，页面组件放在 `src/page` 下：

```text
src/
├─ page/
│  ├─ Login/
│  ├─ Article/
│  ├─ Layout/
│  └─ NotFound/
├─ router/
│  └─ index.js
├─ App.js
└─ index.js
```

路由表集中管理，页面组件只负责自身 UI 和业务逻辑，后期扩展权限、懒加载和嵌套路由会更清晰。

## 四、最小路由应用

React Router 6+ 可以用数据路由方式创建路由：

```jsx
// src/router/index.js
import { createBrowserRouter } from 'react-router-dom'

const router = createBrowserRouter([
  {
    path: '/login',
    element: <div>登录页</div>,
  },
  {
    path: '/article',
    element: <div>文章页</div>,
  },
])

export default router
```

在入口文件通过 `RouterProvider` 注入：

```jsx
import React from 'react'
import ReactDOM from 'react-dom/client'
import { RouterProvider } from 'react-router-dom'
import router from './router'

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <RouterProvider router={router} />
  </React.StrictMode>
)
```

`RouterProvider` 接收路由实例，负责根据地址匹配并渲染路由元素。

## 五、抽象路由模块

实际项目中不要把所有页面都写成内联 `<div>`，可以导入页面组件：

```jsx
import { createBrowserRouter } from 'react-router-dom'
import Login from '../page/Login'
import Article from '../page/Article'
import NotFound from '../page/NotFound'

const router = createBrowserRouter([
  { path: '/login', element: <Login /> },
  { path: '/article', element: <Article /> },
  { path: '*', element: <NotFound /> },
])

export default router
```

路由配置只负责“路径 → 页面”的映射，页面自身的请求、状态和交互仍由组件管理。路径最好使用稳定、语义清晰的命名，避免把页面逻辑堆在路由文件中。

## 六、声明式导航：`Link` 与 `NavLink`

声明式导航通过组件属性描述“要去哪里”，适合菜单、面包屑和普通页面链接：

```jsx
import { Link, NavLink } from 'react-router-dom'

function Header() {
  return (
    <nav>
      <Link to="/">首页</Link>
      <Link to="/article">文章</Link>
      <NavLink
        to="/about"
        className={({ isActive }) => (isActive ? 'active' : '')}
      >
        关于
      </NavLink>
    </nav>
  )
}
```

`Link` 最终会渲染为浏览器支持的链接，但由 React Router 接管跳转，通常不会触发整页刷新。`NavLink` 额外提供 `isActive`，适合需要当前菜单高亮的导航。不要用普通 `<a href>` 替代站内路由跳转，否则可能重新加载整个应用。

## 七、编程式导航：`useNavigate`

当跳转由事件、登录结果或业务判断触发时，可以使用 `useNavigate`：

```jsx
import { Link, useNavigate } from 'react-router-dom'

function Login() {
  const navigate = useNavigate()

  const handleLogin = async () => {
    // 登录请求成功
    navigate('/article')
  }

  return (
    <div>
      <Link to="/article">声明式跳转</Link>
      <button onClick={handleLogin}>登录并跳转</button>
      <button onClick={() => navigate(-1)}>返回上一页</button>
    </div>
  )
}
```

`navigate('/path')` 跳到指定路径，`navigate(-1)` 返回历史记录，`navigate('/path', { replace: true })` 可以替换当前历史记录，适合登录成功后不希望用户返回登录页的场景。`useNavigate` 只能在 Router 管理的组件中调用。

## 八、导航传参：路径参数与查询参数

### 1. 路径参数 params

在路由 path 中使用 `:参数名` 定义动态片段：

```jsx
{
  path: '/article/:id/:name',
  element: <Article />,
}
```

跳转时把值放在路径中：

```jsx
navigate('/article/1001/jack')
```

页面中使用 `useParams` 读取：

```jsx
import { useParams } from 'react-router-dom'

function Article() {
  const { id, name } = useParams()
  return <div>文章：{id} - {name}</div>
}
```

### 2. 查询参数 search params

查询参数放在 `?` 后面：

```jsx
navigate('/article?id=1001&name=jack')
```

使用 `useSearchParams` 读取：

```jsx
import { useSearchParams } from 'react-router-dom'

function Article() {
  const [params] = useSearchParams()
  const id = params.get('id')
  const name = params.get('name')
  return <div>文章：{id} - {name}</div>
}
```

路径参数适合资源身份，例如文章 id；查询参数适合筛选、排序、分页等可选条件。读取到的值通常是字符串，需要按业务转换成数字或布尔值。

## 九、嵌套路由与布局组件

一级路由中继续嵌套子路由，就形成嵌套路由。公共导航、侧边栏和页脚可以放在父级布局组件中，子页面放在 `Outlet` 位置。

```jsx
// router/index.js
const router = createBrowserRouter([
  {
    path: '/',
    element: <Layout />,
    children: [
      { index: true, element: <Board /> },
      { path: 'about', element: <About /> },
    ],
  },
])
```

```jsx
// page/Layout/index.js
import { Link, Outlet } from 'react-router-dom'

function Layout() {
  return (
    <div>
      <nav>
        <Link to="/">面板</Link>
        <Link to="/about">关于</Link>
      </nav>
      <main>
        <Outlet />
      </main>
    </div>
  )
}
```

父级路由负责公共外壳，`Outlet` 决定子路由渲染位置。子路由的 path 通常写相对路径，例如 `about`，不要重复写开头的 `/`。

## 十、默认二级路由：`index`

访问 `/` 时，如果父路由只配置了 `children` 但没有默认子路由，`Outlet` 可能没有内容。可以使用 `index: true` 指定默认子路由：

```jsx
{
  path: '/',
  element: <Layout />,
  children: [
    {
      index: true,
      element: <Board />,
    },
    {
      path: 'about',
      element: <About />,
    },
  ],
}
```

`index` 子路由不需要再写 `path`，它匹配父路由本身的地址。一个父路由下应根据页面结构明确设置默认内容，避免进入布局后出现空白区域。

## 十一、404 兜底路由

当用户输入的路径没有匹配任何路由时，应展示友好的 404 页面：

```jsx
const router = createBrowserRouter([
  { path: '/', element: <Layout />, children: [...] },
  { path: '/login', element: <Login /> },
  { path: '*', element: <NotFound /> },
])
```

`*` 可以匹配所有未命中的路径，通常放在路由表末尾。NotFound 页面可以提供返回首页的 `Link`，也可以根据需要显示帮助信息。

```jsx
function NotFound() {
  return (
    <div>
      <h1>页面不存在</h1>
      <Link to="/">返回首页</Link>
    </div>
  )
}
```

## 十二、两种路由模式

React Router 常见的路由模式有 history 和 hash：

| 模式 | URL 表现 | 实现基础 | 刷新是否需要后端支持 |
| --- | --- | --- | --- |
| history | `/login` | History API、`pushState` | 需要服务器把未知路径回退到入口页 |
| hash | `/#/login` | `hashchange` | 通常不需要额外配置 |

创建方式：

```jsx
import { createBrowserRouter, createHashRouter } from 'react-router-dom'

const historyRouter = createBrowserRouter(routes)
const hashRouter = createHashRouter(routes)
```

history 地址更简洁，但部署时要配置服务器回退规则；hash 兼容静态服务器更方便，但 URL 中会出现 `#`。选择模式时要结合部署环境，而不是只看开发阶段的地址是否好看。

## 十三、路由表完整示例

下面的配置串联了布局、默认页、关于页、登录页、参数页和 404：

```jsx
import { createBrowserRouter } from 'react-router-dom'
import Layout from '../page/Layout'
import Board from '../page/Board'
import About from '../page/About'
import Login from '../page/Login'
import Article from '../page/Article'
import NotFound from '../page/NotFound'

const router = createBrowserRouter([
  {
    path: '/',
    element: <Layout />,
    children: [
      { index: true, element: <Board /> },
      { path: 'about', element: <About /> },
    ],
  },
  { path: '/login', element: <Login /> },
  { path: '/article/:id/:name', element: <Article /> },
  { path: '*', element: <NotFound /> },
])

export default router
```

建议先画出 URL 和页面组件的对应表，再写路由配置；嵌套关系明确后，`Outlet` 的位置和公共布局也会更容易设计。

## 十四、记账本项目的路由准备

课程后续的记账本项目会在路由基础上组织多个页面。可以先按职责规划：

```text
/                 → 记账本首页
/new              → 新增账单
/detail/:id       → 账单详情
/statistics       → 统计页面
/*                → 404 页面
```

如果多个页面共享顶部导航和底部操作栏，可以使用 `Layout` 作为父路由，具体页面作为 children。列表筛选、月份、分类等条件适合通过查询参数表达，例如 `/statistics?month=2024-01`。

路由只负责页面切换和参数解析，账单数据的保存、请求和状态管理应交给组件状态、Redux 或后续项目的数据层。

## 十五、常见错误与排查顺序

1. **页面没有渲染**：确认入口文件使用 `RouterProvider`，并且传入的是正确的 router 实例。
2. **二级路由空白**：父组件是否渲染了 `<Outlet />`，children 的 path 是否写成相对路径。
3. **导航刷新整页**：站内跳转使用 `Link`/`NavLink`，不要用普通 `<a href>`。
4. **`useNavigate` 报错**：确认组件处于 RouterProvider 管理的组件树中。
5. **参数取不到**：path 参数名要和 `useParams` 解构名一致，查询参数要通过 `useSearchParams` 读取。
6. **首页进入空白**：在父路由 children 中配置 `index: true`。
7. **404 不生效**：确认配置了 `path: '*'`，且没有被错误的父路由结构挡住。
8. **history 部署刷新 404**：服务器需要把未知前端路径回退到应用入口 HTML。

排查时先看地址栏是否改变，再看路由表是否命中，最后检查布局中的 `Outlet` 和参数读取逻辑。

## 十六、本日总结

- 前端路由把 URL path 映射到页面 component。
- `createBrowserRouter`/`createHashRouter` 创建路由，`RouterProvider` 注入应用。
- `Link` 和 `NavLink` 用于声明式导航，`useNavigate` 用于编程式导航。
- `useParams` 读取路径参数，`useSearchParams` 读取查询参数。
- `children` 配置嵌套路由，`Outlet` 指定子页面渲染位置，`index` 配置默认子路由。
- `path: '*'` 提供 404 兜底。
- history 地址简洁但需要服务器回退，hash 兼容静态部署但地址带 `#`。

下一步可以在路由基础上实现记账本的页面切换、账单详情和统计页面，并继续学习路由级数据加载和权限控制。